# 08 · Cost per successful call

## 1 — Learning contract
By the end of this notebook you will be able to:
1. Estimate the dollar cost of one call by hand from **turns × per-turn unit prices**
   (LLM + TTS + STT), and say which turns cost what.
2. Compute **cost per successful call** = total estimated spend ÷ number of **completed** calls,
   and explain why a failed call still *spends* money but adds **nothing** to the denominator.
3. Compare a **clean** cohort against a **messy** cohort and show the messy one costs more
   per success even when each individual call looks cheap.
4. Attach the words **"estimated, prototype"** to every number you show, and defend the cost
   model as honest-by-construction (it lives in `schemas/cost.md`).

Everything here is arithmetic on counts you already have — no model calls, no new data.
The lesson is not the multiplication; it is *what you divide by*.

## 2 — Knowledge map (where this book sits)

`07 (failure tags) → THIS: cost per successful call → 09 (language conditions)`

In **07** you learned to tag *why* a call failed (barge-in, kb-gap, language mismatch). That gave
you a label on each broken call. This book turns those labels into a **money number**: a failed
call is not just a red tag, it is **spend with no completed task behind it** — it shrinks the
denominator of every cost-per-success figure your dashboard shows.

Why this book exists, and exists *here*: founders do not fund "the barge-in rate dropped." They
fund "each successful booking costs us 4 cents instead of 11." Next door, **09** holds language
fixed and varies the condition — and you will see a Hinglish/Telugu call's *extra* turns show up
directly as *extra cost per success*. Cost is the unit that makes every earlier signal spendable.

## 3 — Baby intuition

Imagine a vending machine that charges you **per button press**, not per snack. Every press costs
a fixed few cents — whether or not a snack actually drops.

A voice agent is that machine. Each **turn** (the agent speaking, the caller speaking) presses
buttons in three boxes: the **ear** (STT, turning speech into text), the **brain** (LLM, deciding
the reply), and the **mouth** (TTS, speaking the reply). You pay per press.

Now the cruel part: if the call ends with **no booking** — the caller hung up confused — you still
paid for every press. You bought nothing. So the honest cost of a *working* agent is not "cost per
call." It is **cost per call that actually worked**, and the broken calls quietly raise that price.

## 4 — The formal version

Two numbers, computed in this order.

**(a) Cost of ONE call** — multiply turns by unit prices and add the boxes up:

> `call_cost = (agent_turns × llm_price) + (agent_turns × tts_price) + (user_turns × stt_price)`

The agent's turns drive the **LLM** (it has to think) and the **TTS** (it has to speak). The user's
turns drive the **STT** (we transcribe what they said). Turns are the meter; unit prices are the
rate.

**(b) Cost per SUCCESSFUL call** — over a cohort of calls:

> `cost_per_success = (sum of every call's cost) ÷ (number of calls whose task COMPLETED)`

The numerator counts **all** spend, success or failure. The denominator counts **only successes**.
That asymmetry is the whole idea: a failed call adds to the top and nothing to the bottom, so it
**raises** the price of the successes that remain.

| term | meaning |
|---|---|
| **unit price** | dollars per turn for one box (LLM / TTS / STT) — a *toy estimate* here |
| **call cost** | total estimated dollars for one call |
| **completed call** | `task_completed == True` (the required-fields checklist passed — book 06) |
| **cost per success** | cohort spend ÷ completed-call count |
| **estimated, prototype** | the label that rides on every figure (`schemas/cost.md`) |

## 5 — Why this exists (the part founders care about)

"The agent is cheap" is a feeling. **"Each completed booking costs 3.8 cents to produce, and our
messy cohort costs 9 cents because a third of those calls fail"** is a business case — a number you
can put a target on, watch move, and prove you improved.

This is also the book where every earlier signal becomes **spendable**. A barge-in (book 04) that
forces the caller to repeat themselves is not just rude — it adds turns, and turns are money. A
failure tag (book 07) is not just a label — it removes a call from the success denominator. Cost is
the common currency the whole pipeline pays into.

One honesty rule, stated now and enforced in every cell: these are **toy unit prices** and an
**estimated** model. We never print a cost without the words *estimated, prototype* — the cost
schema (`schemas/cost.md`) literally carries a disclaimer field so no rendering can drop the caveat.
We start where the course always starts: a single call, costed by hand, before any function.

## CHECKPOINT 1 (say it out loud before continuing)
1. Write the cost-per-success formula from memory. What goes in the **numerator**, and what is the
   one rule about what goes in the **denominator**?
2. Which box (LLM / TTS / STT) is driven by **agent** turns, and which by **user** turns?
3. A call that fails costs money but adds nothing below the line. In one sentence: what does that do
   to the cost per success of the calls that *did* work?

## ACT 1 knowledge-flow checkpoint — what changed in your head?

Before Act 1 you (probably) thought: a cheaper agent is one whose calls cost less. After Act 1 you
should hold: cost is **turns × unit price**, and the number that matters is **cost per *successful*
call** — where failed calls spend from the top while adding nothing to the bottom, quietly raising
the price of every success. Cost is a measurement, not a mood.

If that feels like your own sentence, continue. If not, re-read the formal table in cell 4.

In [ ]:
# YOUR TURN - learning log, Act 1. Write YOUR one-sentence version of why "cost per call" is the
# wrong number and "cost per SUCCESSFUL call" is the right one. Producing the sentence is the
# learning; reading mine would only feel like it.
clean_sentence_act_1 = ""   # <- type your sentence between the quotes

# A gentle gate so you cannot skim past: the cell nags until you write something real.
if len(clean_sentence_act_1.strip()) < 15:
    print("write your Act-1 sentence above (15+ characters), then re-run this cell.")
else:
    print("ACT 1 LOGGED:", clean_sentence_act_1)

# Act 2 — Mechanics: cost one call by hand, then the whole cohort

## Toy unit prices, printed RAW first

Course rule: the ugly input goes on screen **before** anything is computed from it. Here are the
**toy** per-turn unit prices we will use all book. They are round, made-up estimates — chosen so
you can do the arithmetic in your head, not pulled from a vendor invoice. (`estimated, prototype`.)

In [ ]:
# Toy per-turn unit prices in US dollars. We keep them as a plain dict and PRINT them raw before
# using them - you should be able to see the rate card before any bill is computed from it.
# These are deliberately round toy estimates so the by-hand math stays visible; not a real quote.
unit_prices = {
    "llm_per_agent_turn": 0.0030,   # the "brain": one agent turn = one LLM reply to generate
    "tts_per_agent_turn": 0.0010,   # the "mouth": one agent turn = one spoken reply to synthesize
    "stt_per_user_turn":  0.0005,   # the "ear":   one user turn  = one chunk of speech to transcribe
}
for box, price in unit_prices.items():
    print(f"{box:<22} ${price:.4f} per turn   (estimated, prototype)")

## One toy call — meet Call A (clean English success)

We start with the cleanest member of the recurring cast: **Call A**, an English booking, cooperative
caller, low turns, task completed. We will count its turns by speaker, then cost it by hand.

In [ ]:
# Call A as a tiny toy: just the fields cost needs (id, language, outcome, and the turns with a
# speaker each). We print it RAW so the input is visible before any counting - seeing the ugly
# input first is a course rule, and cost is only ever as honest as the turn counts feeding it.
call_A = {
    "call_id": "call_A",
    "language": "English",
    "task_completed": True,            # the required-fields checklist passed (book 06)
    "turns": [
        {"speaker": "agent", "text": "Hi, I can book that. What date works?"},
        {"speaker": "user",  "text": "Tuesday morning, please."},
        {"speaker": "agent", "text": "Booked Tuesday 10am. Anything else?"},
        {"speaker": "user",  "text": "No, thanks."},
    ],
}
for t in call_A["turns"]:
    print(f"{t['speaker']:<6} | {t['text']}")

## PREDICT
Call A has **4 turns**: agent, user, agent, user.
1. How many turns are **agent** turns, and how many are **user** turns?
2. Using the rate card above, will the **LLM** part or the **STT** part cost more for this call?
Commit before the next cell.

In [ ]:
# YOUR TURN - lock your prediction BEFORE the counting cell runs, so the notebook records YOUR
# thinking and a later cell can compare it against reality. That comparison is the lesson.
my_agent_turn_guess = None   # <- replace None with how many turns are 'agent'
my_user_turn_guess  = None   # <- replace None with how many turns are 'user'

if my_agent_turn_guess is None or my_user_turn_guess is None:
    print("fill in BOTH guesses above, then re-run this cell.")
else:
    print("locked:", my_agent_turn_guess, "agent turns,", my_user_turn_guess, "user turns")

In [ ]:
# Count turns by speaker BY HAND - a plain loop, no library shortcut, because the whole cost number
# rests on these two counts and you should watch them be built one turn at a time.
agent_turns = 0
user_turns = 0
for t in call_A["turns"]:
    # we branch on the speaker because agent turns drive LLM+TTS while user turns drive STT -
    # the speaker IS the cost driver, so we must keep the two counts separate, not just total turns
    if t["speaker"] == "agent":
        agent_turns += 1
    elif t["speaker"] == "user":
        user_turns += 1
print("agent turns:", agent_turns, "| user turns:", user_turns)

# the metal-detector reading: did YOUR committed prediction match?
if my_agent_turn_guess is not None:
    ok = (my_agent_turn_guess == agent_turns and my_user_turn_guess == user_turns)
    print("your guess", "matched" if ok else "DIFFERED",
          "- if it differed, that gap between your model and reality is the thing to chew on")

## PREDICT
Now the money. With **2 agent turns** and **2 user turns**, and the rate card
(LLM \$0.0030, TTS \$0.0010 per agent turn; STT \$0.0005 per user turn):
1. What is the LLM cost? The TTS cost? The STT cost?
2. What is the **total** call cost, in dollars? Commit a number before the next cell.

In [ ]:
# YOUR TURN - lock the total-cost prediction before the compute cell reveals it.
my_call_A_cost = None   # <- your by-head total for Call A, in dollars (e.g. 0.009)

if my_call_A_cost is None:
    print("fill in my_call_A_cost above, then re-run this cell.")
else:
    print("locked: $", my_call_A_cost, "predicted for Call A")

In [ ]:
# Cost ONE call BY HAND, every box shown separately - no helper function yet (that comes after you
# can do this by hand). We split into three lines because the whole point is that cost is the SUM of
# three independent meters, each = (count of the right turns) x (its unit price).
llm_cost = agent_turns * unit_prices["llm_per_agent_turn"]   # brain: charged per agent turn
tts_cost = agent_turns * unit_prices["tts_per_agent_turn"]   # mouth: charged per agent turn
stt_cost = user_turns  * unit_prices["stt_per_user_turn"]    # ear:   charged per user turn

call_A_cost = llm_cost + tts_cost + stt_cost                 # the bill is the three meters added up
print(f"LLM ${llm_cost:.4f} + TTS ${tts_cost:.4f} + STT ${stt_cost:.4f} = ${call_A_cost:.4f}  (estimated, prototype)")

if my_call_A_cost is not None:
    print("your prediction", "matched" if round(my_call_A_cost, 4) == round(call_A_cost, 4) else "DIFFERED")

## EXPLAIN gate
One sentence, out loud, in this shape:
> "Call A cost about ___ cents because it had ___ agent turns (LLM+TTS) and ___ user turns (STT)."

(About 0.9 cents. Notice the LLM line alone — \$0.006 — is two-thirds of the bill. The brain is the
expensive box; that fact will matter when we compare clean vs messy calls.)

## Manual-before-function — now wrap it, once you trust it

You costed Call A by hand and watched the three meters. **Only now** do we wrap that exact
arithmetic in a function, so we can run it over many calls without retyping. The function does
nothing your hand did not — it just stops you fat-fingering the multiplication on call number forty.

In [ ]:
# A small, honest cost function: it is literally the three lines you just ran by hand, returned as a
# dict so the breakdown is never lost behind a single total. We pass unit_prices in (never hardcode
# them inside) because prices are an estimate you will want to change, not a fact baked into code.
def cost_one_call(call, prices):
    # recount turns inside the function so it works on ANY call, not just Call A - cost must be a
    # pure function of (this call's turns) x (the rate card), with no leftover state from earlier cells
    a = sum(1 for t in call["turns"] if t["speaker"] == "agent")
    u = sum(1 for t in call["turns"] if t["speaker"] == "user")
    llm = a * prices["llm_per_agent_turn"]
    tts = a * prices["tts_per_agent_turn"]
    stt = u * prices["stt_per_user_turn"]
    return {"agent_turns": a, "user_turns": u, "total": llm + tts + stt}

# confirm the function reproduces the by-hand number exactly - a wrapper that disagrees with the
# hand math is a bug, so we check before trusting it on the rest of the book
check = cost_one_call(call_A, unit_prices)
print("function says:", round(check["total"], 4), "| by hand was:", round(call_A_cost, 4),
      "->", "match" if round(check["total"], 4) == round(call_A_cost, 4) else "MISMATCH")

## CHECKPOINT 2 (out loud, without scrolling up)
1. The cost formula for one call, from memory: which turns drive LLM+TTS, and which drive STT?
2. Why does `cost_one_call` take `prices` as an argument instead of using the numbers inside it?
3. The LLM line was the biggest part of Call A's bill. What does that predict about a call that
   has the agent repeat itself three extra times?

## Meet the other two cast members — B (Hinglish partial) and C (Telugu failure)

The recurring cast travels the whole course. We now bring in **Call B** (Hinglish, hesitations and
a repeat, task **partially** done) and **Call C** (Telugu-English, the agent interrupts mid-answer,
language mismatch, task **failed**). Watch their turn counts — and their `task_completed` flags.

In [ ]:
# Calls B and C, same toy shape. The point of the cast: B and C take MORE turns to get LESS done.
# We keep task_completed honest to the spec (A success, B partial->not completed, C failure) because
# that flag is exactly what the cost-per-SUCCESS denominator will count later.
call_B = {
    "call_id": "call_B",
    "language": "Hinglish",
    "task_completed": False,           # partial: a repeat was needed, one field never confirmed
    "turns": [
        {"speaker": "agent", "text": "Aap kis date pe chahte ho?"},
        {"speaker": "user",  "text": "umm... shayad Tuesday? ya Wednesday..."},
        {"speaker": "agent", "text": "Sorry, Tuesday or Wednesday?"},          # the repeat = extra turns
        {"speaker": "user",  "text": "Tuesday, Tuesday."},
        {"speaker": "agent", "text": "Theek hai. Morning ya evening?"},
        {"speaker": "user",  "text": "morning... I think"},
    ],
}
call_C = {
    "call_id": "call_C",
    "language": "Telugu-English",
    "task_completed": False,           # failure: barge-in + address ambiguity, nothing booked
    "turns": [
        {"speaker": "agent", "text": "Mee area cheppandi?"},
        {"speaker": "user",  "text": "Madhapur side, near the er--"},
        {"speaker": "agent", "text": "Madhapur, okay, and the date--"},        # barge-in: cut them off
        {"speaker": "user",  "text": "no no, not Madhapur, Manikonda"},
        {"speaker": "agent", "text": "Sorry, which area?"},
        {"speaker": "user",  "text": "Manikonda! M-A-N-I..."},
        {"speaker": "agent", "text": "I think we got disconnected earlier, can you repeat?"},
        {"speaker": "user",  "text": "(sighs) forget it"},
    ],
}
for c in (call_B, call_C):
    print(c["call_id"], "|", c["language"], "| turns:", len(c["turns"]), "| completed:", c["task_completed"])

## PREDICT
Three calls now: A (4 turns, success), B (6 turns, not completed), C (8 turns, not completed).
1. Rank them by **individual call cost**, cheapest to most expensive.
2. Does the most expensive call get the *most* done? Commit before running.

In [ ]:
# YOUR TURN - lock your cheapest->priciest ranking before the cohort cell.
# Write the three ids in order, cheapest first, as a list of strings.
my_cost_ranking = None   # e.g. ["call_A", "call_B", "call_C"]

if my_cost_ranking is None:
    print("fill in my_cost_ranking above (a list of 3 ids), then re-run.")
else:
    print("locked ranking (cheapest first):", my_cost_ranking)

In [ ]:
# Cost ALL THREE calls with the function we trust, and print each one's bill. We keep the cohort as a
# list so the next step (summing spend, counting successes) is a loop over it - the same shape the
# real analytics layer uses over a folder of calls.
cohort = [call_A, call_B, call_C]
billed = []
for c in cohort:
    info = cost_one_call(c, unit_prices)
    billed.append({"call_id": c["call_id"], "completed": c["task_completed"], **info})
    # printing per-call so the RAW bills are visible before we aggregate - aggregates lie when you
    # cannot see the rows that fed them (the lesson of the whole stress act, previewed here)
    print(f"{c['call_id']:<8} turns(a/u)={info['agent_turns']}/{info['user_turns']} "
          f"cost=${info['total']:.4f}  completed={c['task_completed']}   (estimated, prototype)")

## OBSERVE + EXPLAIN
Call C is the **most expensive** call — and it booked **nothing**. Call A is the **cheapest** — and
it is the only success. One sentence: why does individual call cost, on its own, point you in
exactly the wrong direction about value?

## PREDICT — the number that matters
Now the cohort-level number. Total spend is the sum of all three bills. But the denominator counts
**only completed** calls — and only **A** completed.
1. What is the total spend across A + B + C (roughly, in cents)?
2. How many completed calls is the denominator? (Count the `True` flags.)
3. So what is **cost per successful call**? Commit before the next cell.

In [ ]:
# YOUR TURN - lock all three before the reveal: total spend, # completed, cost-per-success.
my_total_spend     = None   # sum of the three bills, in dollars
my_n_completed     = None   # how many of A/B/C have task_completed == True
my_cost_per_success = None  # total spend / n_completed

if any(v is None for v in (my_total_spend, my_n_completed, my_cost_per_success)):
    print("fill in all three (spend, n_completed, cost-per-success) above, then re-run.")
else:
    print("locked  spend $", my_total_spend, " completed:", my_n_completed,
          " per-success $", my_cost_per_success)

In [ ]:
# THE central computation of the book. Numerator: ALL spend (success or failure). Denominator: ONLY
# completed calls. We compute them on separate lines so the asymmetry is impossible to miss - the
# same total is divided by a SMALLER number the more calls fail.
total_spend = sum(b["total"] for b in billed)               # every call's cost, no exceptions
n_completed = sum(1 for b in billed if b["completed"])      # only the successes go below the line
cost_per_success = total_spend / n_completed                # spend per call that actually worked

print(f"total spend (all 3 calls):  ${total_spend:.4f}")
print(f"completed calls (denominator): {n_completed}   <- of 3 attempted")
print(f"cost per SUCCESSFUL call:    ${cost_per_success:.4f}   (estimated, prototype)")
print(f"naive cost per CALL:         ${total_spend/len(billed):.4f}   <- divides by 3, the wrong number")

if my_cost_per_success is not None:
    print("your per-success", "matched" if round(my_cost_per_success, 4) == round(cost_per_success, 4) else "DIFFERED")

## OBSERVE
The naive "cost per call" divides by **3** and looks cheap. The honest "cost per **successful**
call" divides by **1** — because B and C, despite costing the *most*, produced no completed task.
The two failures did not just waste their own money; they **tripled the price** of A's success. That
gap between the two numbers is the entire reason this book exists.

## CHECKPOINT 3 (out loud, without scrolling up)
1. State the numerator and the denominator of cost-per-success, and which calls each one counts.
2. The naive "cost per call" was much lower than "cost per successful call." Which one would you
   show a founder, and why is showing only the cheap one a quiet lie?
3. Call C cost the most and completed nothing. Explain its full effect on cost-per-success in one
   sentence (hint: it touches *both* the top and the bottom of the fraction — but only the top).

## ACT 2 knowledge-flow checkpoint — what changed in your head?

Before: cost was a single per-call number. After Act 2 you can cost one call by hand
(turns × unit price across three boxes), wrap it in a function you trust, and then compute the
number that actually matters — **cost per successful call** — where the numerator counts all spend
and the denominator counts only completions. You watched two pricey failures triple the price of one
cheap success. The arithmetic is trivial; the choice of denominator is the lesson.

In [ ]:
# YOUR TURN - learning log, Act 2.
clean_sentence_act_2 = ""   # your one-liner for Act 2 (turns x price, or the denominator choice - your pick)

if len(clean_sentence_act_2.strip()) < 15:
    print("write your Act-2 sentence above, then re-run.")
else:
    print("ACT 2 LOGGED:", clean_sentence_act_2)

# Act 3 — Stress: clean vs messy, the averaging trap, and breaking the cost model

## Two cohorts: clean vs messy

The single number "cost per successful call" earns its keep when you **compare two cohorts**. We
build a **clean** cohort (mostly short successes) and a **messy** cohort (longer calls, more
failures) — and watch the messy one cost more per success even though each messy call, taken alone,
does not look alarming.

In [ ]:
# Build two toy cohorts as lists of (turns, completed) - we only need turn counts and the success
# flag to cost a cohort, so we keep these stripped down. agent/user split is fixed at half-and-half
# here so the comparison isolates ONE thing: how messiness (more turns, more failures) moves cost.
def make_call(call_id, n_turns, completed):
    # alternate agent/user so a call of n turns has a clean half-and-half split - this keeps the
    # comparison about turn COUNT and success RATE, not about who happened to talk more
    turns = [{"speaker": "agent" if i % 2 == 0 else "user", "text": ""} for i in range(n_turns)]
    return {"call_id": call_id, "task_completed": completed, "turns": turns}

# clean cohort: short calls, almost all succeed (one stumble)
clean_cohort = [make_call(f"clean_{i}", n, ok) for i, (n, ok) in enumerate([
    (4, True), (4, True), (6, True), (4, True), (6, False),
])]
# messy cohort: longer calls (repeats, barge-ins add turns), a THIRD of them fail
messy_cohort = [make_call(f"messy_{i}", n, ok) for i, (n, ok) in enumerate([
    (10, True), (12, False), (8, True), (14, False), (10, True), (12, False),
])]
print("clean cohort:", [(c["call_id"], len(c["turns"]), c["task_completed"]) for c in clean_cohort])
print("messy cohort:", [(c["call_id"], len(c["turns"]), c["task_completed"]) for c in messy_cohort])

In [ ]:
# A cohort-level cost-per-success helper: it is exactly the Act-2 computation (sum spend, divide by
# completions), now over a list. We return the parts, not just the ratio, so a caller can SEE the
# numerator and denominator that produced the number - a bare ratio with no parts is unauditable.
def cost_per_success(cohort, prices):
    spend = sum(cost_one_call(c, prices)["total"] for c in cohort)   # numerator: all spend
    completed = sum(1 for c in cohort if c["task_completed"])        # denominator: only successes
    # guard against divide-by-zero: a cohort with ZERO successes has an undefined per-success cost,
    # and inventing a number there would be the exact dishonesty this book argues against
    per_success = spend / completed if completed else None
    return {"spend": spend, "completed": completed, "n_calls": len(cohort), "per_success": per_success}

## PREDICT
Each **messy** call costs more than each clean call (more turns). And the messy cohort has a worse
success rate (a third fail vs one in five). So when we divide:
1. Will the messy cohort's **cost per success** be a little higher, or a lot higher?
2. Will the gap be bigger or smaller than the gap in *per-call* cost? Commit before running.

In [ ]:
# YOUR TURN - lock which cohort costs more per success, and your guess for the ratio.
my_pricier_cohort = None   # "clean" or "messy"
my_per_success_ratio = None  # messy per-success / clean per-success, your rough guess (e.g. 2.0)

if my_pricier_cohort is None or my_per_success_ratio is None:
    print("fill in both (which cohort, and the ratio guess) above, then re-run.")
else:
    print("locked:", my_pricier_cohort, "pricier, ratio ~", my_per_success_ratio)

In [ ]:
# Compare the two cohorts head to head. We print spend, completions, per-call, and per-success side
# by side so the DIVISOR's effect is visible - the per-success gap is wider than the per-call gap,
# and that widening is caused entirely by the messy cohort's smaller success denominator.
clean = cost_per_success(clean_cohort, unit_prices)
messy = cost_per_success(messy_cohort, unit_prices)

for name, r in (("clean", clean), ("messy", messy)):
    per_call = r["spend"] / r["n_calls"]
    print(f"{name:<6} spend ${r['spend']:.4f} | {r['completed']}/{r['n_calls']} completed "
          f"| per-call ${per_call:.4f} | per-success ${r['per_success']:.4f}   (estimated, prototype)")

ratio = messy["per_success"] / clean["per_success"]
print(f"\nmessy costs {ratio:.1f}x clean PER SUCCESS  (vs only "
      f"{(messy['spend']/messy['n_calls'])/(clean['spend']/clean['n_calls']):.1f}x per call)")
if my_pricier_cohort is not None:
    print("your pick", "matched" if my_pricier_cohort == "messy" else "DIFFERED")

## OBSERVE
The per-*call* cost gap is modest — messy calls are only somewhat longer. But the per-*success* gap
is wider, because the messy cohort spends on calls that never make it below the line. Failures are a
**denominator tax**: they do not just cost their own turns, they make every success around them more
expensive. That is why a chart of "cost per call" can look fine while the business quietly bleeds.

## WRONG-INTUITION TRAP — "cheaper per call means cheaper to run"

**The wrong belief:** "Cohort X has a lower average cost per call, so it is the cheaper agent to
operate."

We will build two cohorts where the trap springs hard: cohort LOW has a **lower cost per call** but
a **higher cost per success**, because its cheap calls are cheap precisely *because they fail fast*.
Run it, then try to explain the reversal BEFORE reading the reveal.

In [ ]:
# Cohort LOW: many SHORT calls, but most FAIL (they end early - cheap, because nothing got done).
# Cohort HIGH: fewer, LONGER calls, but most SUCCEED. We engineer this so per-call and per-success
# DISAGREE about which is cheaper - the exact situation a per-call dashboard hides.
low_cohort = [make_call(f"low_{i}", n, ok) for i, (n, ok) in enumerate([
    (2, False), (2, False), (2, False), (2, False), (4, True),   # cheap because they fail fast
])]
high_cohort = [make_call(f"high_{i}", n, ok) for i, (n, ok) in enumerate([
    (8, True), (8, True), (10, True), (8, False),                # pricier calls, but they work
])]

low = cost_per_success(low_cohort, unit_prices)
high = cost_per_success(high_cohort, unit_prices)
print(f"LOW : per-call ${low['spend']/low['n_calls']:.4f}  | per-success ${low['per_success']:.4f}  ({low['completed']}/{low['n_calls']} ok)")
print(f"HIGH: per-call ${high['spend']/high['n_calls']:.4f}  | per-success ${high['per_success']:.4f}  ({high['completed']}/{high['n_calls']} ok)")
print("\nper-CALL says LOW is cheaper.  per-SUCCESS says HIGH is cheaper.  Same dollars, opposite verdict.")

## The reveal

Cohort LOW wins on **cost per call** and loses on **cost per success** — and per-success is the one
tied to value produced. LOW's calls are cheap *because they fail*: a call that hangs up after two
turns spends almost nothing and books nothing. Optimizing for low cost-per-call rewards exactly the
behavior you least want — **failing quickly and cheaply**.

This is the same shape as the averaging traps earlier in the course: a number can be real, computed
correctly, render beautifully — **and answer the wrong question**. "Cost per call" answers *"how
much per attempt?"*; "cost per success" answers *"how much per result?"*. Only the second one is the
business. Report per-call alone and you will congratulate an agent for getting good at giving up.

In [ ]:
# YOUR TURN - explain the trap in your own words, stored for future-you.
my_trap_explanation = ""   # one sentence: why can a LOWER cost-per-call hide a HIGHER cost-per-success?

if len(my_trap_explanation.strip()) < 20:
    print("write your explanation above (20+ chars), then re-run.")
else:
    print("TRAP EXPLAINED:", my_trap_explanation)

## CHECKPOINT 4 (out loud)
A founder is shown two agents: agent LOW at \$0.001/call, agent HIGH at \$0.003/call. They pick LOW.
In one sentence each: what number did the slide hide, what behavior does "minimize cost per call"
secretly reward, and which single number would have flipped the decision?

## BREAK-IT (guided) — the zero-success cohort

Real cohorts can have a stretch where **nothing** completes (a bad deploy, a broken integration).
Cost per success = spend ÷ completions — so what is the cost per success when completions is **0**?
Predict first: does our code **crash**, return **infinity**, or refuse to answer?

## PREDICT
We cost a cohort where **every** call failed (`completed == 0`). The denominator is zero.
Does `cost_per_success` divide by zero and crash, or did we guard it? And what *should* an honest
cost report say when there are no successes to divide by?

In [ ]:
# BREAK-IT (guided): a cohort with ZERO successes. We predict the guard returns None rather than
# crashing or faking a number - "cost per success" is genuinely UNDEFINED with no successes, and the
# only honest output is to say so, not to invent a finite-looking dollar figure.
all_fail_cohort = [make_call(f"fail_{i}", n, False) for i, n in enumerate([6, 8, 4])]
r = cost_per_success(all_fail_cohort, unit_prices)
print("spend (real money was still burned):", round(r["spend"], 4))
print("completed:", r["completed"])
print("cost per success:", r["per_success"], "<- None means UNDEFINED, not free")
# the spend is NON-zero (those calls cost money) but per-success is None - the agent spent and
# produced nothing, which is the most expensive state of all, and our code refuses to hide it

## Reading the result — None is the honest answer

The spend is real and non-zero; the per-success is `None`. That `None` is not a bug — it is the
truthful statement *"this cohort produced no successes, so there is no cost-per-success to report,
and the spend was pure loss."* A cohort that burned money and completed nothing is the **most**
expensive outcome possible, and inventing a finite dollar figure (or silently dividing by zero into
`inf`) would disguise that. When you cannot divide, say so — never fake the denominator.

## BREAK-IT (learner-authored) — your own damage to the cost model

Author your own break. Pick **one** thing to corrupt and predict its effect on the cost number:
- make a unit price **negative** (a vendor "credit") and watch cost go down — is that real savings?
- give a call **zero turns** (an empty/dropped call) — what is its cost, and is "free" honest?
- flip a failed call's `task_completed` to `True` — watch cost-per-success drop without any real
  improvement (the classic way to lie with this metric).

Write your prediction as a comment, then run.

In [ ]:
# YOUR TURN - self-authored BREAK-IT on the cost model.
# my prediction: <write here exactly what you expect to happen to the cost number, and why>
import copy

my_prices = copy.deepcopy(unit_prices)     # damage a COPY of the rate card, not the real one
my_cohort = copy.deepcopy(messy_cohort)    # and a copy of a cohort, so later cells stay intact

# 1) damage ONE thing here (uncomment and edit ONE line):
# my_prices["llm_per_agent_turn"] = -0.0030          # a 'credit' that makes cost fall - real, or fake?
# my_cohort[0]["turns"] = []                          # an empty call: zero turns -> $0 cost
# my_cohort[1]["task_completed"] = True               # relabel a failure as a success (the metric lie)

# 2) recompute and read the cost number against your written prediction:
r = cost_per_success(my_cohort, my_prices)
print("spend:", round(r["spend"], 4), "| completed:", r["completed"],
      "| per-success:", None if r["per_success"] is None else round(r["per_success"], 4))

## CHECKPOINT 5 (out loud)
1. When a cohort has zero successes, what does `cost_per_success` return, and why is that more
   honest than `inf` or a made-up number?
2. Relabeling one failed call as completed drops cost-per-success with **zero** real improvement.
   Name the guardrail that stops this (hint: it is in book 06 — who decides `task_completed`?).
3. A negative unit price made cost fall. Is that a real efficiency win, or a change to the
   *estimate*? Which word on every figure reminds you which it is?

## ACT 3 knowledge-flow checkpoint — what changed in your head?

Before: a lower cost-per-call felt like a cheaper agent. After Act 3: failures are a **denominator
tax** that inflate cost-per-success; per-call cost can rank cohorts in exactly the wrong order
(rewarding fast cheap failure); a zero-success cohort has an **undefined** (not free) cost; and the
two easiest ways to lie with this metric are relabeling failures as successes and quietly editing
the unit-price estimate. The math is honest only when the denominator and the "estimated" label are.

In [ ]:
# YOUR TURN - learning log, Act 3.
clean_sentence_act_3 = ""   # your one-liner (the denominator tax, or 'per-call lies', your pick)

if len(clean_sentence_act_3.strip()) < 15:
    print("write your Act-3 sentence above, then re-run.")
else:
    print("ACT 3 LOGGED:", clean_sentence_act_3)

# Act 4 — Ownership: the real schema, the chart, and defending the dollars

## Where this lives in VoiceForge — `schemas/cost.md`

Nothing today was a metaphor. The cost record is a real schema, **honest by construction**:

| field (from `schemas/cost.md`) | what it is | what you built today |
|---|---|---|
| `est_cost_total` | turns × per-turn LLM/TTS/STT estimate | `cost_one_call(...)["total"]` |
| `est_llm_calls` | ≈ agent turns | your `agent_turns` count |
| `turn_count` | total turns | `len(call["turns"])` |
| `est_cost_per_success_note` | fixed string `"estimated, prototype"` | the label on every print |

The schema carries that note **as a field** so no dashboard rendering can strip the caveat. And the
spec is explicit: *"cost per successful call is computed at the aggregate level: total estimated cost
of a cohort ÷ number of completed-task calls; failed calls make the denominator smaller — that is
the business-value chart."* That sentence is exactly the computation you ran in Act 2.

We now load the **real** normalized calls and cost them with the same function — toy prices, real
turn counts.

In [ ]:
# Load the REAL normalized calls. We resolve the path by walking up from the working directory so the
# notebook runs whether the kernel started in notebooks/ or the repo root (no hardcoded abs path).
import json
from pathlib import Path

norm_dir = next(p for p in [Path.cwd()/"data"/"normalized",
                            *[a/"data"/"normalized" for a in Path.cwd().parents]] if p.exists())
real_calls = [json.loads(p.read_text()) for p in sorted(norm_dir.glob("*.json"))]
# we print the shape FIRST (how many calls, their languages) before costing any - know the cohort
# you are holding before you bill it, the table-reading ritual applied to a folder of calls
print("real normalized calls:", len(real_calls))
for c in real_calls[:3]:
    print(" ", c["call_id"], "| lang", c["language"], "| turns", len(c["turns"]), "| stress", c["stress_profile"])

## PREDICT
The real calls range from 12 turns (the hero call) to 46 turns (a long SpokenWOZ dialogue).
1. Which real call will be the **most expensive** to run, and why?
2. The normalized files carry no `task_completed` flag yet — so what would you need before you could
   compute cost per **successful** call on real data? Commit before running.

In [ ]:
# YOUR TURN - lock your guess for the priciest real call.
my_priciest_real = None   # the call_id you think costs the most (hint: most turns)

if my_priciest_real is None:
    print("fill in my_priciest_real above (a call_id string), then re-run.")
else:
    print("locked:", my_priciest_real)

In [ ]:
# Cost every real call with the SAME function and toy prices. Real turn counts, estimated rates.
# We sort by cost so the priciest surfaces by itself rather than being eyeballed, and we keep the
# turn count beside the cost so the turns->dollars relationship stays visible on real data.
real_billed = []
for c in real_calls:
    info = cost_one_call(c, unit_prices)
    real_billed.append({"call_id": c["call_id"], "turns": len(c["turns"]), "cost": info["total"]})

for b in sorted(real_billed, key=lambda x: -x["cost"]):
    print(f"{b['call_id']:<14} turns={b['turns']:>3}  est_cost=${b['cost']:.4f}   (estimated, prototype)")

priciest = max(real_billed, key=lambda x: x["cost"])
print("\npriciest real call:", priciest["call_id"], f"(${priciest['cost']:.4f}, {priciest['turns']} turns)")
if my_priciest_real is not None:
    print("your guess", "matched" if my_priciest_real == priciest["call_id"] else "DIFFERED")

## Cost per success on real data — with a HONEST caveat

The normalized pool does not ship a `task_completed` flag (that comes from the task-outcome layer,
book 06). To show the real cost-per-success computation, we attach a **clearly-labeled assumed**
outcome per call — derived from its `stress_profile` as a stand-in — and we *say so loudly*. The
arithmetic is identical to Act 2; only the source of the success flag is a placeholder.

In [ ]:
# Attach an ASSUMED success flag from stress_profile as a stand-in for the real task-outcome layer.
# This is a placeholder, declared as one - 'clean' calls are assumed completed, the harder stress
# profiles assumed not. We would replace this with the real task_completed in production; the point
# here is the SHAPE of the computation, not these specific success labels.
assumed_completed = lambda c: c["stress_profile"] == "clean"   # stand-in ONLY; not ground truth

real_cohort = []
for c in real_calls:
    real_cohort.append({"call_id": c["call_id"], "turns": c["turns"],
                        "task_completed": assumed_completed(c)})

r = cost_per_success(real_cohort, unit_prices)
print(f"ASSUMED-success cohort (placeholder flags, NOT ground truth):")
print(f"  spend ${r['spend']:.4f} | completed {r['completed']}/{r['n_calls']} | "
      f"per-success ${r['per_success']:.4f}   (estimated, prototype)")

## PREDICT — the chart
We will draw two bars: **cost per call** vs **cost per successful call** for the messy cohort from
Act 3. Before the chart: which bar is taller, and roughly how many times taller?

In [ ]:
# Two bars: per-call vs per-success for the messy cohort. We chart these two side by side because the
# whole book is the GAP between them - the chart's single licensed claim is "per-success > per-call,
# and the failures are what open that gap." Every line says why it exists.
import matplotlib.pyplot as plt

m = cost_per_success(messy_cohort, unit_prices)
labels = ["cost per CALL", "cost per SUCCESS"]              # x: the two ways to divide (THINGS)
values = [m["spend"] / m["n_calls"], m["per_success"]]      # y: dollars under each divisor (MEASURE)

fig, ax = plt.subplots(figsize=(5, 3))   # fig = canvas, ax = the drawing area on it
ax.bar(labels, values)                   # one bar per divisor; height = dollars
ax.set_ylabel("estimated $ (prototype)") # the caveat rides even on the axis label, by duty
ax.set_title("messy cohort: same spend, two divisors")
for i, v in enumerate(values):           # print the value on each bar so the gap is legible, not guessed
    ax.text(i, v, f"${v:.4f}", ha="center", va="bottom")
plt.show()

## Read the chart — the 4-question ritual
1. **x?** the two divisors (per call vs per success). 2. **y?** estimated dollars. 3. **one bar?**
the cost under one choice of denominator. 4. **what does it license?** Exactly one claim: dividing
the *same* spend by completions (not by attempts) raises the price, and the size of that rise **is**
the cost of the failures. It does **not** license "the agent got more expensive" — the spend never
moved; only the denominator did.

## The three-level explanation (same concept, three rooms)

- **To a beginner:** "Every back-and-forth turn costs a little money — to listen, to think, to
  speak. We add that up per call, then divide by only the calls that actually worked. Broken calls
  still cost money but count for nothing, so they make each good call look more expensive."
- **To an engineer:** "`call_cost = agent_turns·(llm+tts) + user_turns·stt`, summed over a cohort
  for the numerator; denominator = `count(task_completed)`. Failures inflate the ratio without
  touching the numerator's per-call mean. Per-call cost is a misleading proxy (it rewards fast
  failure); report cost-per-success. Zero successes → undefined, not `inf`. All figures carry
  `est_cost_per_success_note = 'estimated, prototype'` per `schemas/cost.md`."
- **To a founder:** "We know what one *successful outcome* costs to produce — not one attempt, one
  result. It is 3.8 cents on clean traffic and rises on messy traffic because failures spend money
  for nothing. It is an estimate on prototype prices, labeled as such, and it is the number we set a
  target on and prove down."

## Defense questions (×3 — try first, then open)

**1. "These prices are made up — so isn't the whole cost number fiction?"**
<details><summary>answer</summary>The unit prices are toy estimates and every figure says so
("estimated, prototype" — it is a field in `schemas/cost.md`, not a footnote we can drop). But the
*structure* is real and price-independent: cost scales with turns, and the per-success ratio always
rises with failures. Swap in a vendor's real rate card and the conclusions hold; only the cents
change.</details>

**2. "Why cost per *successful* call instead of plain cost per call?"**
<details><summary>answer</summary>Because cost per call rewards the wrong thing: a call that fails
fast is cheap, so "minimize cost per call" pushes the agent to give up quickly. Cost per success
ties spend to value produced — failed calls spend from the numerator but add nothing to the
denominator, which is exactly the business reality. We showed two cohorts where per-call and
per-success disagree on which is cheaper.</details>

**3. "Could someone game this metric?"**
<details><summary>answer</summary>Yes — relabel a failed call as completed and cost-per-success
drops with no real improvement. That is why `task_completed` is a deterministic required-fields
checklist (book 06), not a self-reported flag, and why a zero-success cohort returns undefined
rather than a flattering number. The metric is only as honest as the success definition feeding its
denominator.</details>

## ACT 4 knowledge-flow checkpoint — what changed in your head?

You should now own the whole loop: real turn counts × toy unit prices → per-call cost → cohort spend
÷ completions → cost per success → a two-bar chart that licenses exactly one claim. You can place
every piece in `schemas/cost.md`, attach "estimated, prototype" to every figure by reflex, and
defend why the denominator — not the multiplication — is where the honesty lives. Next door (09),
language conditions add turns, and you now have the unit to price that in: cost per success.

## TEACH-BACK GATE — the pass bar for this book

**Close this notebook.** Two minutes, out loud, no peeking. Hit all five:
1. The one-call cost formula: which turns drive LLM+TTS, which drive STT.
2. Cost per successful call: the numerator (all spend) and the denominator (completions only).
3. Why two pricey failures tripled the price of one cheap success (the denominator tax).
4. The trap: how a *lower* cost-per-call can hide a *higher* cost-per-success (fast cheap failure).
5. What a zero-success cohort returns, and why "estimated, prototype" rides on every figure.

Could not hit all five? Open it back up, find the gap, redo that act. That is the system working.

In [ ]:
# YOUR TURN - final learning log: Act 4 sentence + YOUR clean sentence for the whole book.
clean_sentence_act_4 = ""   # one-liner for Act 4 (the real schema / defending the dollars)
my_clean_sentence = ""      # the sentence you would say in a room about voice cost

if len(clean_sentence_act_4.strip()) < 15 or len(my_clean_sentence.strip()) < 15:
    print("write both sentences above, then re-run.")
else:
    print("ACT 4 LOGGED:", clean_sentence_act_4)
    print("YOUR CLEAN SENTENCE:", my_clean_sentence)

## The book's clean sentence (read only after writing yours)

> **"Voice quality is a money number, not a feelings number."**

The cast proved it: Call C *felt* bad and *was* expensive — most turns, zero completions, and it
tripled the price of Call A's one success. One multiplication per call and one honest denominator
turned "the agent feels wasteful" into "each successful booking costs 3.8 cents, rising on messy
traffic." If your sentence captures that in your own words, this book did its job.

In [ ]:
# SELF-AUDIT - counts this notebook's own structure from the .ipynb on disk.
# A claim of compliance is wet cement; this counts. (Same logic as notebooks/audit_nb.py.)
import json, re
from pathlib import Path
name = "08_cost_per_successful_call.ipynb"   # <- this notebook's filename
nb_path = next(p for p in [Path.cwd()/name, Path.cwd()/"notebooks"/name,
               *[a/"notebooks"/name for a in Path.cwd().parents]] if p.exists())
nb = json.loads(nb_path.read_text())
S = lambda c: c["source"] if isinstance(c["source"], str) else "".join(c["source"])
pool = [c for c in nb["cells"] if "SELF-AUDIT" not in S(c) and "banned phrase" not in S(c).lower()]
md = [c for c in pool if c["cell_type"]=="markdown"]; co = [c for c in pool if c["cell_type"]=="code"]
cnt = lambda m, g: sum(1 for c in g if m in S(c))
reason = lambda c: any(len(l.strip("# ").split())>=4 for l in S(c).splitlines() if l.strip().startswith("#"))
t = " ".join(S(c).lower() for c in pool)
checks = {
 "total cells 50-90": (len(nb["cells"]), 50<=len(nb["cells"])<=90),
 "specific checkpoints >=5": (cnt("CHECKPOINT", md), cnt("CHECKPOINT", md)>=5),
 "act knowledge-flow cps >=4": (cnt("knowledge-flow checkpoint", md), cnt("knowledge-flow checkpoint", md)>=4),
 "predict prompts >=8": (cnt("PREDICT", pool), cnt("PREDICT", pool)>=8),
 "break-it >=2": (cnt("BREAK-IT", co)+cnt("EXPECTED FAILURE FOR LEARNING", co), cnt("BREAK-IT", co)+cnt("EXPECTED FAILURE FOR LEARNING", co)>=2),
 "wrong-intuition trap >=1": (cnt("WRONG-INTUITION TRAP", md), cnt("WRONG-INTUITION TRAP", md)>=1),
 "learner cells >=6": (cnt("YOUR TURN", co), cnt("YOUR TURN", co)>=6),
 "reasoning comments all": (sum(map(reason, co)), all(map(reason, co)) if co else False),
 "3-level explanation": (1, all(w in t for w in ("beginner","engineer","founder"))),
 "teach-back": (cnt("TEACH-BACK", md), cnt("TEACH-BACK", md)>=1),
 "clean sentence": (1, "clean sentence" in t),
 "banned phrases =0": (sum(bool(re.search(r"\b"+w+r"\b", t)) for w in ["obviously","as you know","simply","intuitively"]), not any(re.search(r"\b"+w+r"\b", t) for w in ["obviously","as you know","simply","intuitively"])),
}
print(f"{'metric':<28}{'n':>5}  verdict"); ok=True
for k,(n,p) in checks.items():
    ok &= p; print(f"{k:<28}{n:>5}  {'PASS' if p else 'FAIL'}")
print("AUDIT:", "ALL PASS - now do the teach-back" if ok else "FAIL")

## Next on the ladder

**08 done** (pending your teach-back) → **09 · Language conditions** — you now have the unit that
prices everything: cost per successful call. In 09 you hold the task fixed and vary the language
condition (English vs Hinglish vs Telugu-English), and the extra turns a code-switched call needs
show up directly as **extra cost per success** — the number you just learned to compute.